In [14]:
import os
import re
import zipfile
import requests
from datetime import datetime
from bs4 import BeautifulSoup

In [15]:
import time
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.transport.requests import Request

In [ ]:
# Main is at the end

# BASE DIRECTORY SETUP
BASE_DIR = os.getcwd()
DOWNLOAD_DIR = os.path.join(BASE_DIR, "raw_download")
TARGET_EXTRACT_DIR = os.path.join(DOWNLOAD_DIR, "qblox_docs_latest")
OUTPUT_FILE = os.path.join(BASE_DIR, "qblox_lean_docs_latest.txt")

TOKEN_FILE = os.path.join(BASE_DIR, "token.json")
CREDENTIALS_FILE = os.path.join(BASE_DIR, "credentials.json")

In [17]:
def get_live_version_marker():
    """
    Queries the public Qblox landing page using a browser signature
    to bypass CDN security walls. Returns a server tracking signature.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
    }
    try:
        response = requests.get(PUBLIC_HOMEPAGE, headers=headers, timeout=10)
        if response.status_code == 200:
            marker = response.headers.get('ETag', '') + response.headers.get('Last-Modified', '')
            if not marker:
                marker = response.text[:500] 
            return marker[:50].strip()
        else:
            print(f"Public page check failed. Server returned status code: {response.status_code}")
            return None
    except Exception as e:
        print(f"Network error checking Qblox public server: {e}")
        return None

def download_and_extract_docs():
    """Downloads the raw HTML zip compilation from the active track."""
    if not os.path.exists(DOWNLOAD_DIR):
        os.makedirs(DOWNLOAD_DIR)
        
    zip_file_path = os.path.join(DOWNLOAD_DIR, "qblox_raw.zip")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    print(f"Downloading core document archive from: {QBLOX_ZIP_URL}")
    response = requests.get(QBLOX_ZIP_URL, headers=headers, stream=True, allow_redirects=True)
    
    if response.status_code == 200:
        with open(zip_file_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete.")
        
        print(f"Extracting raw structures to: {TARGET_EXTRACT_DIR}")
        if not os.path.exists(TARGET_EXTRACT_DIR):
            os.makedirs(TARGET_EXTRACT_DIR)
            
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(TARGET_EXTRACT_DIR)
            
        os.remove(zip_file_path) # Drop temporary archive zip
        print("Extraction complete! File trees are built for text extraction.")
        return True
    else:
        print(f"Download thread halted. Server returned status: {response.status_code}")
        return False

In [18]:
def parse_and_clean_docs():
    """
    Surgically extracts core Qblox API & Q1ASM syntax from HTML trees.
    Aggressively filters directory structures, redundant module boilerplate, 
    API parameter tables, and structural markdown artifacts.
    """
    print("Executing standalone element text extraction and deep-clean...")
    
    if not os.path.exists(TARGET_EXTRACT_DIR):
        print(f"Error: Target directory {TARGET_EXTRACT_DIR} does not exist.")
        return False

    # STEP 2 IMPLEMENTATION: Semantic Content Filtering via Directory Isolation
    # Only walk files explicitly mapping to core API/Native layouts; ignore /tutorials or /examples
    html_files = []
    for root, dirs, files in os.walk(TARGET_EXTRACT_DIR):
        # Drop non-essential directories immediately from the walk tree
        if any(folder in root.lower() for folder in ["tutorials", "examples", "assets"]):
            continue
            
        for file in files:
            # Target absolute source-of-truth frameworks and skip generic prose pages
            if file.endswith(".html"):
                html_files.append(os.path.join(root, file))
                
    if not html_files:
        print("Error: Could not find any target HTML framework files.")
        return False

    # Configuration Guardrails (Optimized Sets for O(1) Lookups where applicable)
    SECTION_BLACKLIST = {
        "installation", "handling", "servicing", "placement", "grounding", 
        "sma connections", "ethernet connection", "unusual conditions", 
        "swapping modules", "swapping or removing modules", "fuses", 
        "batteries", "connecting the spi components", "connecting to power", 
        "product compliance", "product compliance information", "regulatory standards (safety)", 
        "regulatory standards (emc)", "environmental characteristics", "mainframe front", 
        "mainframe back", "power supply", "power rating", "physical dimensions", 
        "absolute maximum ratings", "intended use", "environmental", "power chain", 
        "battery unit", "gyrator", "external power supply unit"
    }

    PHYSICAL_FLUFF_KEYWORDS = [
        "unboxing guidelines", "facility maintenance", "unpacking", "desktop device",
        "mounted into a 19” rack", "indoor use only", "air in- and outlets",
        "free from obstruction", "cooling fans", "ventilation requirements",
        "interfere with the ventilation", "overheating", "dynamic fan control",
        "conducting and grounded", "conductive and grounded", "grounding plates",
        "braided cable", "braided ground cable", "star configuration",
        "ground loops", "ground currents", "mains ground", "galvanically isolated",
        "disconnecting device", "power switch", "power cord", "power supply unit",
        "mains socket", "fuse tray", "time-delayed fuse", "short circuit",
        "torque", "maximum torque", "torque wrench", "torque screwdriver",
        "tightening the screws", "retighten the top", "loose screws", "unscrewing",
        "black handle", "white button", "horizontal position", "grip the module",
        "thumb and your fingers", "top and bottom edges", "slide the module",
        "metal casing", "slits", "empty slots", "flowblockers", "wrist strap",
        "esd safe", "foot grounder", "conductive floor", "electrostatically discharged",
        "soft cloth", "cleaning", "clean the inside", "clean the outside",
        "emits noise, smell, or sparks", "hot when extracted", "dangerous", "high-voltage parts",
        "warranty", "voids the warranty", "opened under any circumstances",
        "substitute parts", "modification to the product",
        "pollution degree", "overvoltage category", "max altitude", "max humidity",
        "ip rating", "ip 20", "ik rating", "ik 08", "class a", "class b",
        "rohs", "en 61010-1", "en iec 63000", "en 55011", "en 61326-1", "fcc part 15",
        "residential environments", "radio reception", "receiving antenna", "class b digital device",
        "lead-acid batteries", "ah total", "choke and active circuitry"
    ]

    CORE_ENGINEERING_MARKERS = [
        "q1asm", "sequencer", "register", "0x", "def ", "import ", "api", 
        "parameter", "cluster", "qcm", "qrm", "qtm", "waveform", "pulse",
        "gain", "offset", "frequency", "phase", "synq", "linq", "scpi"
    ]

    # Seen set to prevent massive duplication of identical modules text (AWG, Mixer correction, etc.)
    seen_content_hashes = set()
    clean_lines = []

    for target_file in sorted(html_files):
        with open(target_file, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f, "html.parser")

        # Extract main semantic document body
        article = soup.find("article", class_="bd-article") or soup.find("body") or soup
        
        # Decompose layout & metadata artifacts early
        for element in article(["script", "style", "button", "noscript", "form", "footer", 
                                ".prev-next-area", "metadata", "rdf:rdf", "sodipodi:namedview"]):
            element.decompose()

        for element in article.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p", "li", "tr"]):
            # Deduplication Guard (Child Skip)
            if element.name in ["p", "li"] and element.find_parent(["li", "tr"]):
                continue
                
            text = element.get_text().strip()
            if not text:
                continue
                
            text_lower = text.lower()
            
            # Drop low-level Inkscape layout data strings
            if any(token in text_lower for token in ["sodipodi:", "inkscape:", "pagecolor=", "bordercolor="]):
                continue

            # Process Section Headers
            if element.name in ["h1", "h2", "h3", "h4", "h5", "h6"]:
                if not any(blacklist_item in text_lower for blacklist_item in SECTION_BLACKLIST):
                    # Ensure unique headings to help suppress Duplicate Module Boilerplate
                    if text_lower not in seen_content_hashes:
                        seen_content_hashes.add(text_lower)
                        clean_lines.append(f"\n\n# {text}\n")
                continue

            # CULPRIT 1 FIX: Content Deduplication Guard
            # Generates a quick hash of structural prose phrases to avoid repeating 
            # exact AWG / Gain / Offset modules explanations across QRM, QCM, and QRC.
            content_hash = hash(text_lower)
            if content_hash in seen_content_hashes:
                continue

            # Filter Core Content vs Physical Fluff
            if any(marker in text_lower for marker in CORE_ENGINEERING_MARKERS):
                seen_content_hashes.add(content_hash)
                clean_lines.append(text)
            elif any(keyword in text_lower for keyword in PHYSICAL_FLUFF_KEYWORDS):
                continue
            else:
                # Catch-all for regular text blocks ensuring uniqueness
                seen_content_hashes.add(content_hash)
                clean_lines.append(text)
    
    # Combine content stream
    raw_output = "\n".join(clean_lines)

    # STEP 1 IMPLEMENTATION: High-Speed Regex Pipeline Cleanup
    print("Executing final regex-based syntax and artifact scrub...")
    # 1. Clear hyperlinked Jupyter notebook download lines
    raw_output = re.sub(r"A Jupyter notebook version of this tutorial can be downloaded here\.", "", raw_output, flags=re.IGNORECASE)
    # 2. Scrub Plug & Play terminal commands and common setup markdown remnants
    raw_output = re.sub(r"!qblox-pnp list|#\s*Setup\s*#|#\s*Scan For Clusters\s*#", "", raw_output, flags=re.IGNORECASE)
    # 3. Clean escaping artifacts left behind by HTML-to-text transitions
    raw_output = raw_output.replace("\\_", "_").replace("\\[", "[").replace("\\]", "]")
    # 4. Collapse high-density empty bracket structures / empty lists
    raw_output = re.sub(r"\[\s*\]|\{\s*\}", "", raw_output)
    # 5. Compress multiple empty lines/line breaks down to standard spacing
    compressed_text = re.sub(r'\n{3,}', '\n\n', raw_output)

    # Write Final Production File
    current_date = datetime.now().strftime("%Y_%m_%d")
    with open(OUTPUT_FILE, "w", encoding="utf-8") as outfile:
        outfile.write(f"=========================================\n")
        outfile.write(f"QBLOX CORE HARDWARE & API KNOWLEDGE BASE\n")
        outfile.write(f"SYNCHRONIZED ON: {current_date}\n")
        outfile.write(f"=========================================\n\n")
        outfile.write(compressed_text)

    final_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)
    print(f"\nSuccess! Optimized text document generated: {OUTPUT_FILE}")
    print(f"Final File Size: {final_size_mb:.4f} MB")
    return True

In [19]:
def authenticate_google_drive():
    """Handles silent OAuth authentication or prompts browser login if expired."""
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())
            except Exception:
                if os.path.exists(TOKEN_FILE):
                    os.remove(TOKEN_FILE)
                flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
                creds = flow.run_local_server(port=0)
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open(TOKEN_FILE, 'w') as token:
            token.write(creds.to_json())
            
    return creds


def get_stored_cloud_marker(drive_service, file_name):
    """Retrieves the execution tracking signature directly from Google Drive metadata description."""
    query = f"name = '{file_name}' and mimeType = 'application/vnd.google-apps.document' and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, description)").execute()
    items = results.get('files', [])
    if items:
        return items[0].get('id'), items[0].get('description', '').strip()
    return None, ""


def upload_to_google_drive(local_file_path, live_marker):
    """Splits payload into 3 balanced parts and updates Google Docs atomically in reverse order."""
    try:
        creds = authenticate_google_drive()
        drive_service = build('drive', 'v3', credentials=creds)
        docs_service = build('docs', 'v1', credentials=creds)
        
        with open(local_file_path, 'r', encoding='utf-8') as f:
            local_text_content = f.read()
            
        lines = local_text_content.splitlines(keepends=True)
        total_lines = len(lines)
        lines_per_file = (total_lines // 3) + 1
        
        file_payloads = [
            "".join(lines[0:lines_per_file]),
            "".join(lines[lines_per_file:lines_per_file * 2]),
            "".join(lines[lines_per_file * 2:])
        ]
        
        target_files = [
            "Sir_Qblox_Master_Docs_Part1",
            "Sir_Qblox_Master_Docs_Part2",
            "Sir_Qblox_Master_Docs_Part3"
        ]
        
        for part_idx, file_name in enumerate(target_files):
            print(f"\n--- Synchronizing {file_name} ---")
            payload = file_payloads[part_idx]
            
            file_id, _ = get_stored_cloud_marker(drive_service, file_name)
            
            if file_id:
                print(f"Permanent Google Doc found (ID: {file_id}). Wiping content...")
                doc = docs_service.documents().get(documentId=file_id).execute()
                end_index = doc.get('body').get('content')[-1].get('endIndex') - 1
                
                if end_index > 1:
                    wipe_req = [{'deleteContentRange': {'range': {'startIndex': 1, 'endIndex': end_index}}}]
                    docs_service.documents().batchUpdate(documentId=file_id, body={'requests': wipe_req}).execute()
                    time.sleep(1.5)
            else:
                print(f"Creating permanent placeholder asset for {file_name}...")
                file_metadata = {'name': file_name, 'mimeType': 'application/vnd.google-apps.document'}
                if DRIVE_FOLDER_ID:
                    file_metadata['parents'] = [DRIVE_FOLDER_ID]
                new_file = drive_service.files().create(body=file_metadata, fields='id').execute()
                file_id = new_file.get('id')
                print(f"Established! ID: {file_id}.")

            CHUNK_SIZE = 100000
            total_chars = len(payload)
            chunks = [payload[i:i + CHUNK_SIZE] for i in range(0, total_chars, CHUNK_SIZE)]
            print(f"Compiling {len(chunks)} chunks into a reverse-ordered stack payload layout...")
            
            # CORE FIX: Loop backwards over chunks, writing to index 1. 
            # This completely preserves chronological reading order over Google's JSON batch mechanism.
            compiled_requests = []
            for chunk in reversed(chunks):
                compiled_requests.append({
                    'insertText': {
                        'location': {'index': 1},
                        'text': chunk
                    }
                })
            
            if compiled_requests:
                print(f" -> Transmitting structural payload transaction to cloud...")
                docs_service.documents().batchUpdate(documentId=file_id, body={'requests': compiled_requests}).execute()
                print(f" -> Successfully committed all text components!")
            
            # Write tracking version stamp into Part 1's description metadata
            if part_idx == 0 and live_marker:
                drive_service.files().update(fileId=file_id, body={'description': live_marker}).execute()
                print(f" -> Stamped tracking signature to file card description: [{live_marker}]")
                
            time.sleep(1.5)
                
        print("\nPipeline execution highly successful! All 3 Master Docs are fully synchronized.")
        return True
        
    except Exception as error:
        print(f"An error occurred during the multi-file chunk sync: {error}")
        return False
def authenticate_google_drive():
    """Handles silent OAuth authentication or prompts browser login if expired."""
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())
            except Exception:
                if os.path.exists(TOKEN_FILE):
                    os.remove(TOKEN_FILE)
                flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
                creds = flow.run_local_server(port=0)
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open(TOKEN_FILE, 'w') as token:
            token.write(creds.to_json())
            
    return creds


def get_stored_cloud_marker(drive_service, file_name):
    """Retrieves the execution tracking signature directly from Google Drive metadata description."""
    query = f"name = '{file_name}' and mimeType = 'application/vnd.google-apps.document' and trashed = false"
    results = drive_service.files().list(q=query, fields="files(id, description)").execute()
    items = results.get('files', [])
    if items:
        return items[0].get('id'), items[0].get('description', '').strip()
    return None, ""


def upload_to_google_drive(local_file_path, live_marker):
    """Splits payload into 3 balanced parts and updates Google Docs atomically in reverse order."""
    try:
        creds = authenticate_google_drive()
        drive_service = build('drive', 'v3', credentials=creds)
        docs_service = build('docs', 'v1', credentials=creds)
        
        with open(local_file_path, 'r', encoding='utf-8') as f:
            local_text_content = f.read()
            
        lines = local_text_content.splitlines(keepends=True)
        total_lines = len(lines)
        lines_per_file = (total_lines // 3) + 1
        
        file_payloads = [
            "".join(lines[0:lines_per_file]),
            "".join(lines[lines_per_file:lines_per_file * 2]),
            "".join(lines[lines_per_file * 2:])
        ]
        
        target_files = [
            "Sir_Qblox_Master_Docs_Part1",
            "Sir_Qblox_Master_Docs_Part2",
            "Sir_Qblox_Master_Docs_Part3"
        ]
        
        for part_idx, file_name in enumerate(target_files):
            print(f"\n--- Synchronizing {file_name} ---")
            payload = file_payloads[part_idx]
            
            file_id, _ = get_stored_cloud_marker(drive_service, file_name)
            
            if file_id:
                print(f"Permanent Google Doc found (ID: {file_id}). Wiping content...")
                doc = docs_service.documents().get(documentId=file_id).execute()
                end_index = doc.get('body').get('content')[-1].get('endIndex') - 1
                
                if end_index > 1:
                    wipe_req = [{'deleteContentRange': {'range': {'startIndex': 1, 'endIndex': end_index}}}]
                    docs_service.documents().batchUpdate(documentId=file_id, body={'requests': wipe_req}).execute()
                    time.sleep(1.5)
            else:
                print(f"Creating permanent placeholder asset for {file_name}...")
                file_metadata = {'name': file_name, 'mimeType': 'application/vnd.google-apps.document'}
                if DRIVE_FOLDER_ID:
                    file_metadata['parents'] = [DRIVE_FOLDER_ID]
                new_file = drive_service.files().create(body=file_metadata, fields='id').execute()
                file_id = new_file.get('id')
                print(f"Established! ID: {file_id}.")

            CHUNK_SIZE = 100000
            total_chars = len(payload)
            chunks = [payload[i:i + CHUNK_SIZE] for i in range(0, total_chars, CHUNK_SIZE)]
            print(f"Compiling {len(chunks)} chunks into a reverse-ordered stack payload layout...")
            
            # CORE FIX: Loop backwards over chunks, writing to index 1. 
            # This completely preserves chronological reading order over Google's JSON batch mechanism.
            compiled_requests = []
            for chunk in reversed(chunks):
                compiled_requests.append({
                    'insertText': {
                        'location': {'index': 1},
                        'text': chunk
                    }
                })
            
            if compiled_requests:
                print(f" -> Transmitting structural payload transaction to cloud...")
                docs_service.documents().batchUpdate(documentId=file_id, body={'requests': compiled_requests}).execute()
                print(f" -> Successfully committed all text components!")
            
            # Write tracking version stamp into Part 1's description metadata
            if part_idx == 0 and live_marker:
                drive_service.files().update(fileId=file_id, body={'description': live_marker}).execute()
                print(f" -> Stamped tracking signature to file card description: [{live_marker}]")
                
            time.sleep(1.5)
                
        print("\nPipeline execution highly successful! All 3 Master Docs are fully synchronized.")
        return True
        
    except Exception as error:
        print(f"An error occurred during the multi-file chunk sync: {error}")
        return False

In [20]:
def main():
    print("Checking Qblox documentation server for updates...")
    live_marker = get_live_version_marker()
    
    if not live_marker:
        print("Could not retrieve server signature. Skipping check.")
        return

    # Set up active drive validation to scan document descriptions
    creds = authenticate_google_drive()
    drive_service = build('drive', 'v3', credentials=creds)
    
    # Retrieve previous version marker straight from Google Cloud metadata
    _, last_processed_marker = get_stored_cloud_marker(drive_service, "Sir_Qblox_Master_Docs_Part1")
    print(f"Server Token: [{live_marker}] | Current Cloud Token: [{last_processed_marker}]")

    # Primary Trigger Check
    if live_marker != last_processed_marker or not os.path.exists(OUTPUT_FILE):
        print("Genuine document modifications identified. Executing processing pipelines...")
        if download_and_extract_docs():
            if parse_and_clean_docs():
                print("\nInitiating Google Cloud sync loop...")
                upload_to_google_drive(OUTPUT_FILE, live_marker)
                print("\nPipeline execution complete. Sir Blox is fully synchronized.")
    else:
        print("Zero modifications found on remote documentation tree. Google Cloud asset match identical.")


if __name__ == "__main__":
    main()

Checking Qblox documentation server for updates...
Server Token: ["2574d65d0094d7905607b71245d48b85"Tue, 26 May 2026] | Current Cloud Token: ["2574d65d0094d7905607b71245d48b85"Tue, 26 May 2026]
Genuine document modifications identified. Executing processing pipelines...
Download complete.
Extracting raw structures to: /Users/aayush.kumar/Desktop/Sir Blox/raw_download/qblox_docs_latest
Extraction complete! File trees are built for text extraction.
Executing standalone element text extraction and deep-clean...
Executing final regex-based syntax and artifact scrub...

Success! Optimized text document generated: /Users/aayush.kumar/Desktop/Sir Blox/qblox_lean_docs_latest.txt
Final File Size: 1.1448 MB

Initiating Google Cloud sync loop...

--- Synchronizing Sir_Qblox_Master_Docs_Part1 ---
Permanent Google Doc found (ID: 1_njLq3wwMuIRlDl6vzkdduQVurAP_NnuBtuu5dQhL98). Wiping content...
Compiling 4 chunks into a reverse-ordered stack payload layout...
 -> Transmitting structural payload trans